# 🎓 Quiz Generator: Summary and Multiple-Choice Questions from a PDF

## Learning Objectives
In this notebook, you will learn:
1. **Preparing PDF text** - Extract text with PyPDF2 and repair the broken spacing PDF extraction produces
2. **A summarize → quiz chain** - Summarize the material for a topic, then write questions from the material and the summary
3. **Quiz questions as structured output** - Typed multiple-choice questions (four options, one answer) with an evidence quote for each
4. **Checking the quiz in code** - Question count, option structure, and whether every evidence quote really appears in the PDF
5. **Rendering the study sheet in code** - Build the final document from the checked data instead of asking the model to rewrite it

## Where this fits
Sequential chains with `RunnablePassthrough.assign` come from `05_prompt_chaining/03_prompt_chaining_lcel.ipynb`, structured output from Module 04. This notebook turns them into a study assistant: give it a PDF and a topic, get a summary and a quiz you can trust.

## Prerequisites
- LCEL chaining and structured output from Modules 04–05 (reading only)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`
- The study material `../assets/prompt_engineering.pdf` (in the repository)

---
## 📦 Step 1: Environment Setup

Load the API key from the project-root `.env`.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: summaries and quiz questions
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 📄 Part 1: Loading and Cleaning the PDF

`PyPDF2` extracts the text of every page. PDFs store positioned words, not sentences, so the raw text of this file has **one word per line** and stray spaces inside words ("model’ s", "effectively ."). The model copes with that, but the evidence check in Part 4 compares strings, so the text is normalized once, here.

In [ ]:
# ============================================================================
# LOAD THE PDF (one string for the whole document)
# ============================================================================
import PyPDF2

def load_pdf(file_path):
    """Load and parse a PDF file."""
    with open(file_path, "rb") as file:
        reader = PyPDF2.PdfReader(file)
        # Extract text from all pages
        study_material = ""
        for page in reader.pages:
            study_material += page.extract_text()
    return study_material

raw_material = load_pdf("../assets/prompt_engineering.pdf")
print(f"Raw text: {len(raw_material):,} characters, {raw_material.count(chr(10)):,} line breaks")
print(repr(raw_material[:120]))

In [ ]:
# ============================================================================
# CLEAN THE EXTRACTED TEXT
# ============================================================================
import re


def clean_pdf_text(text: str) -> str:
    text = " ".join(text.split())                       # one line per word → normal spacing
    text = re.sub(r"(\w)’ s\b", r"\1’s", text)          # "model’ s" → "model’s"
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)         # "effectively ." → "effectively."
    text = text.replace("●", "\n• ")                    # bullet glyphs → list items
    return text


study_material = clean_pdf_text(raw_material)
print(f"Clean text: {len(study_material):,} characters\n")
print(study_material[:600])

---
## ⛓️ Part 2: The Summarize → Quiz Chain

Two steps, one after the other: a summary of the material for the chosen topic, then the quiz, which sees both the material and the summary. `RunnablePassthrough.assign` keeps everything in one dictionary: `study_material`, `topic`, `summary`, `quiz`.

### Key Concepts:
- **Evidence quote**: each question carries a sentence copied word for word from the material that supports its answer — so a reviewer (or code) can verify it
- **`answer_index`** instead of repeating the answer text: the correct option is referenced, not rewritten, so it cannot drift from the options

In [ ]:
# ============================================================================
# QUIZ SCHEMA
# ============================================================================
from pydantic import BaseModel, Field

NUM_QUESTIONS = 5


class Question(BaseModel):
    question: str = Field(description="A multiple-choice question about the topic")
    options: list[str] = Field(description="Exactly 4 answer options, without letters")
    answer_index: int = Field(description="Index (0-3) of the correct option")
    evidence: str = Field(description="One sentence copied word for word from the study material "
                                      "that supports the correct answer")


class Quiz(BaseModel):
    questions: list[Question] = Field(description=f"Exactly {NUM_QUESTIONS} questions")


print(f"✅ Quiz schema: {list(Question.model_fields)}")

In [ ]:
# ============================================================================
# SUMMARY + QUIZ CHAINS, RUN IN SEQUENCE
# ============================================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

summarize_chain = ChatPromptTemplate.from_template(
    """Based on the following study material, write a summary for the given topic.
    The summary must not be more than 10 lines.

    Study material: {study_material}
    Topic: {topic}"""
) | llm | StrOutputParser()

quiz_chain = ChatPromptTemplate.from_template(
    """Act as a study assistant for quiz question generation.
    Write exactly {num_questions} multiple-choice questions on the topic, based only on the
    study material. Use the summary to focus on the most important points.
    For each question, copy one sentence from the study material (not from the summary),
    word for word, as evidence.

    Study material: ```{study_material}```
    Summary: ```{summary}```
    Topic: ```{topic}```"""
) | llm.with_structured_output(Quiz)

study_chain = (
    RunnablePassthrough.assign(summary=summarize_chain)
    | RunnablePassthrough.assign(quiz=quiz_chain)
)

TOPIC = "Prompting techniques and what to avoid when writing prompts"
result = study_chain.invoke({"study_material": study_material, "topic": TOPIC,
                             "num_questions": NUM_QUESTIONS})
print(f"📝 Summary:\n{result['summary']}\n")
print(f"❓ {len(result['quiz'].questions)} questions generated")

---
## ✅ Part 3: Checking the Quiz

Before anyone studies from it, check the quiz in code:
- the requested number of questions, each with four distinct options and a valid `answer_index`
- every evidence quote appears in the (cleaned) PDF text — if not, the question may test something the material never said

In [ ]:
# ============================================================================
# QUIZ CHECKS
# ============================================================================
def normalize(text: str) -> str:
    text = text.lower().replace("’", "'").replace("“", '"').replace("”", '"').replace("•", " ")
    text = re.sub(r"\s+([.,;:!?])", r"\1", text)
    return " ".join(text.split()).strip(" .")


MATERIAL = normalize(study_material)


def evidence_found(evidence: str) -> bool:
    """Every clause of the quote (split at : and ;) with 3+ words must appear in the material.
    Clause-level matching accepts a quote that joins a list's lead-in to one of its items."""
    clauses = [c for c in re.split(r"[:;]", normalize(evidence)) if len(c.split()) >= 3]
    return bool(clauses) and all(normalize(c) in MATERIAL for c in clauses)


def quiz_problems(q: Question) -> list[str]:
    problems = []
    if len(q.options) != 4 or len({o.lower() for o in q.options}) != 4:
        problems.append("needs 4 distinct options")
    if not 0 <= q.answer_index < len(q.options):
        problems.append("answer_index out of range")
    if not evidence_found(q.evidence):
        problems.append("evidence not found in the PDF")
    return problems


quiz = result["quiz"]
print(f"{'✅' if len(quiz.questions) == NUM_QUESTIONS else '❌'} "
      f"{len(quiz.questions)}/{NUM_QUESTIONS} questions")
for i, q in enumerate(quiz.questions, 1):
    problems = quiz_problems(q)
    print(f"{'✅' if not problems else '❌'} Q{i}: {q.question[:70]}"
          + (f"  → {problems}" if problems else ""))

# The same check on a question whose "evidence" the material never contains
invented = Question(question="Which technique needs at least 50 examples?",
                    options=["Zero-shot", "Few-shot", "Chain-of-thought", "Fine-tuning"],
                    answer_index=3, evidence="Fine-tuning always requires at least 50 examples.")
print(f"\nInvented question → {quiz_problems(invented)}")

A question with an ❌ evidence problem is not necessarily wrong — the model may have paraphrased, or quoted its own summary instead of the PDF (the prompt forbids it, and it still happens) — but it is the one to review by hand before the quiz is used.

---
## 🖨️ Part 4: Rendering the Study Sheet

The final study sheet is assembled **in code** from the checked summary and questions. Asking the model to "merge" them into a report (an extra LLM call) would let it reword questions or options after they were checked; a template cannot.

In [ ]:
# ============================================================================
# RENDER THE STUDY SHEET (no LLM call)
# ============================================================================
from IPython.display import Markdown, display

LETTERS = "abcd"
lines = [f"## Topic: {TOPIC}", "", "### Summary", result["summary"], "", "### Quiz Questions"]
for i, q in enumerate(quiz.questions, 1):
    lines.append(f"\n**{i}. {q.question}**\n")
    lines += [f"{LETTERS[j]}) {opt}  " for j, opt in enumerate(q.options)]
    lines.append(f"\n*Answer: {LETTERS[q.answer_index]}) {q.options[q.answer_index]}* — "
                 f"“{q.evidence}”" + ("" if not quiz_problems(q) else "  ⚠️ check by hand"))

display(Markdown("\n".join(lines)))

---
## 📝 Summary

In this notebook, we learned:

### 1. Preparing PDF text
- PDF extraction breaks spacing and lines; normalize once before prompting and before any string check

### 2. A summarize → quiz chain
- `RunnablePassthrough.assign` runs the steps in order and keeps every intermediate value

### 3. Structured quiz questions
- Options, answer index and an evidence quote per question turn a text blob into checkable data

### 4. Checking the quiz
- Count, option structure and "is the evidence really in the PDF" are all checked in code; invented evidence is caught

### 5. Rendering in code
- Assemble the final document from checked data with a template, not with another LLM call

### Next Steps
- This is the last notebook of Module 14. For complete multi-service applications, see [repository 14 (projects)](https://github.com/Sourav692/ai-engineering-demystified-14-projects); to revisit any pattern used here, start from **`03_react_agents/03_react_with_create_agent.ipynb`** (agents) or **`05_prompt_chaining/03_prompt_chaining_lcel.ipynb`** (workflows)